# 12 · A crowd heads for coffee 🚶☕

> **Note**
>
> Best run **locally or in the cloud** — the transport loop takes many small steps and the
> closing *parallel* part needs MPI, neither ideal for in-browser JupyterLite.

Pedestrian flow is a **conservation law** for the crowd density $\rho\in[0,1]$ (1 = shoulder-to-shoulder):
$$ \partial_t \rho + \nabla\!\cdot\bigl(\rho\,v(\rho)\,\hat{\mathbf e}\bigr)=0,
   \qquad v(\rho)=v_{\max}\,(1-\rho). $$

- People **slow down in a crowd** — $v(\rho)\to0$ as $\rho\to1$. This nonlinearity makes the **jams**.
- They head in a **desired direction** $\hat{\mathbf e}$ toward the exit (built in §2).
- **Transport keeps fronts sharp** → **discontinuous Galerkin** is the natural tool (§3).

In [ ]:
# --- Google Colab: install NGSolve on first run (a no-op anywhere else) -------
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install",
                    "ngsolve==6.2.2606", "anywidget"], check=True)

In [ ]:
from netgen.occ import WorkPlane, OCCGeometry
from ngsolve import *
from ngsolve.webgui import Draw
import sys

In [ ]:
def progress(i, n, label="shuffling toward the exit"):
    """A tiny dependency-free queue bar (live frontends only; silent on the static build)."""
    import os
    if os.environ.get("WEBGUI_SCENE_DIR"):
        return
    if (i + 1) % max(1, n // 100) == 0 or i + 1 == n:
        f = int(26 * (i + 1) / n)
        sys.stdout.write(f"\r  {label}… [{'█'*f}{'·'*(26-f)}] {100*(i+1)//n:3d}%"); sys.stdout.flush()
        if i + 1 == n:
            sys.stdout.write("\n")

## 1. The room — geometry & boundary conditions

A lecture room empties toward the coffee:

- **3 rows of 4 tables** (two **touching pairs**), with aisles **left / centre / right**.
- The **audience** is packed in behind the tables; the **speakers** are a small group up front.
- A single **door** in the back wall is the only **exit**; walls *and tables* are **no-flux**.

![A top-down lecture room — three rows of four tables (2+2), audience behind them, speakers up front, a door in the back wall, arrows flowing to the door](https://raw.githubusercontent.com/schruste/ngsum2026-colab/colab/data/pedestrian-room.png)

In [ ]:
W, Hh = 9.0, 6.0
da, db = 3.6, 5.4                                   # the door span on the back (bottom) wall
room = (WorkPlane().MoveTo(0, 0).LineTo(da, 0).LineTo(db, 0).LineTo(W, 0)
        .LineTo(W, Hh).LineTo(0, Hh).Close().Face())
for ry in [1.2, 2.8, 4.4]:                          # 3 rows
    for cx in [1.0, 2.2, 5.6, 6.8]:                 # 4 tables = 2 touching pairs (no gap inside a pair)
        room = room - WorkPlane().MoveTo(cx, ry).Rectangle(1.2, 0.4).Face()
room.edges.name = "wall"                            # walls & tables: no-flux
for e in room.edges:                                # name the door segment "exit"
    if abs(e.center[1]) < 1e-6 and da - 0.05 < e.center[0] < db + 0.05:
        e.name = "exit"
mesh = Mesh(OCCGeometry(room, dim=2).GenerateMesh(maxh=0.2))
print(f"{mesh.ne} elements; boundaries: {set(mesh.GetBoundaries())}")
Draw(mesh)

## 2. The navigation field — which way is *out*?

People head for the **door** and **follow walls** rather than walking into them. A cheap,
realistic field:

- Solve a little **Laplace** problem: door at $\varphi=0$ (Dirichlet), walls insulating ($\partial_n\varphi=0$).
- Walk **downhill**: $\hat{\mathbf e}=-\nabla\varphi/|\nabla\varphi|$.
- The Neumann walls make $\hat{\mathbf e}$ automatically **tangential** at walls and tables — nobody
  walks into furniture, for free.

In [ ]:
u, v = H1(mesh, order=2, dirichlet="exit").TnT()
phi = Solve( grad(u) * grad(v) * dx == 1 * v * dx, u[BND("exit")] == 0)
nav = -grad(phi) / sqrt(grad(phi) * grad(phi) + 1e-3)   # ê, exactly normalized — used directly as a CF
Draw(nav, mesh, "navigation field ê — everyone heads for the door", vectors={"grid_size": 30})

**Streamlines** trace the same field as continuous paths — every route bends around the furniture
and ends at the door:

In [ ]:
import numpy as np, matplotlib.pyplot as plt
nv = GridFunction(VectorH1(mesh, order=1)); nv.Set(nav)            # sample ê on a grid (via a GF)
ROWS, COLS = [1.2, 2.8, 4.4], [1.0, 2.2, 5.6, 6.8]                # table corners, as in §1
in_table = lambda px, py: any(cx <= px <= cx+1.2 and ry <= py <= ry+0.4 for ry in ROWS for cx in COLS)
gx, gy = np.linspace(0.06, W-0.06, 110), np.linspace(0.06, Hh-0.06, 74)
U = np.full((len(gy), len(gx)), np.nan); V = np.full_like(U, np.nan)
for i, yy in enumerate(gy):
    for j, xx in enumerate(gx):
        if not in_table(xx, yy):
            try:
                vv = nv(mesh(xx, yy)); U[i, j], V[i, j] = vv[0], vv[1]
            except Exception:
                pass
fig, ax = plt.subplots(figsize=(6.6, 4.4))
ax.streamplot(gx, gy, U, V, color=np.sqrt(U**2 + V**2), cmap="viridis", density=1.4, linewidth=0.8, arrowsize=0.8)
for ry in ROWS:
    for cx in COLS:
        ax.add_patch(plt.Rectangle((cx, ry), 1.2, 0.4, facecolor="#c9a36a", edgecolor="#7a5c2e", zorder=3))
ax.plot([da, db], [0, 0], color="#2ca02c", lw=4, zorder=4)        # the door
ax.set_xlim(0, W); ax.set_ylim(0, Hh); ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
ax.set_title("navigation field ê — streamlines to the door"); plt.tight_layout()

## 3. The conservation law as a DG scheme

Discretise $\partial_t\rho + \nabla\!\cdot\mathbf F(\rho)=0$, with **flux**
$\mathbf F(\rho)=\rho\,v(\rho)\,\hat{\mathbf e}=v_{\max}\,\rho(1-\rho)\,\hat{\mathbf e}$.

**Per element $K$**, multiply by a test $w$ and integrate the divergence by parts:
$$ \int_K \partial_t\rho\,w \;-\; \int_K \mathbf F(\rho)\!\cdot\!\nabla w \;+\; \int_{\partial K}(\mathbf F\!\cdot\!\mathbf n)\,w \;=\;0. $$

- On an edge $\rho$ is **two-valued** — this cell's $\rho$ and the neighbour's $\rho^{\text{o}}$ —
  so the boundary flux $\mathbf F\!\cdot\!\mathbf n$ is **not single-valued**.
- Replace it by one **numerical flux** $\hat f(\rho,\rho^{\text{o}},\mathbf n)$ shared by both cells,
  so the scheme is **conservative**: what leaves one cell enters the other, exactly.

**Lax–Friedrichs flux** — a central average plus a stabilising jump:
$$ \hat f \;=\; \underbrace{\tfrac12\bigl(\mathbf F(\rho)+\mathbf F(\rho^{\text{o}})\bigr)\!\cdot\!\mathbf n}_{\text{central average}}
   \;-\; \underbrace{\tfrac12\,\alpha\,(\rho^{\text{o}}-\rho)}_{\text{dissipation}} . $$

- $\alpha$ is the **largest wave speed** across the edge, $\alpha=\max|\mathbf F'(\rho)\!\cdot\!\mathbf n|$.
  Here $\mathbf F'(\rho)=v_{\max}(1-2\rho)\,\hat{\mathbf e}$, so $|\mathbf F'\!\cdot\!\mathbf n|\le v_{\max}$ → take $\alpha=v_{\max}$.
- The dissipation term **upwinds** the flux and damps oscillations; with $\alpha$ exactly the wave
  speed it is the *least* dissipation that still keeps the scheme stable.
- **Boundaries**: at the **door** the crowd flows out, $\hat f=\mathbf F(\rho)\!\cdot\!\mathbf n$; every
  other edge is a **wall** with **no flux**, $\hat f=0$.

In [ ]:
order = 2
fes = L2(mesh, order=order)            # a quadratic density per cell, discontinuous | no `dgjumps=True` !
rho, w = fes.TnT()
n = specialcf.normal(2)
vmax = 2.0

def F(r): return vmax * r * (1 - r)                  # flux magnitude along ê

ro, bn = rho.Other(), nav * n
fhat = 0.5 * (F(rho) + F(ro)) * bn - 0.5 * vmax * (ro - rho)            # Lax–Friedrichs
conv = (-F(rho) * (nav * grad(w)) * dx                                  # volume term
        + fhat * (w - w.Other()) * dx(skeleton=True)                   # interior edges
        + F(rho) * bn * w * ds(skeleton=True, definedon=mesh.Boundaries("exit")))   # door outflow
C = BilinearForm(conv, nonassemble=True)             # nonlinear → applied each step, never assembled

We add **Artificial viscosity** for stabilization (there are more sophisticated ways to stabilized; but they make the implemenation/discussion more involved as well):
A touch of $h$-scaled **DG-Laplacian** (interior penalty) damps the wiggles, keeps the front sharp, and lets us stay **robust on a coarse mesh**.

In [ ]:
h = specialcf.mesh_size
visc = h / order
def avg(s, t): return 0.5 * (s + t)
diff = (visc * grad(rho) * grad(w) * dx
        + visc * (-avg(grad(rho), grad(ro)) * n * (w - w.Other())
                  - avg(grad(w), grad(w.Other())) * n * (rho - ro)
                  + 10 * order**2 / h * (rho - ro) * (w - w.Other())) * dx(skeleton=True))
D = BilinearForm(diff, nonassemble=True).Assemble()
M = BilinearForm(rho * w * dx).Assemble()

## 4. Fully explicit time stepping — the DG mass-inverse trick

Last section took convection explicitly but the viscosity **implicitly** (IMEX). Take **both
explicitly** and the only matrix left to invert is the **mass matrix** $M$:
$$ \rho^{n+1} = \rho^{n} - \Delta t\,M^{-1}\bigl(C(\rho^{n}) + D\,\rho^{n}\bigr). $$

- For a **DG / `L2`** space $M$ is **block-diagonal** — each cell's basis lives on that cell
  alone — so $M^{-1}$ is a tiny **per-element** solve: `fes.Mass(1).Inverse()`, no global system.
- **Why bother?** That mass solve is **$O(N)$ and parallelises trivially**, where the IMEX
  factor-and-solve of $M+\Delta t\,D$ **couples the whole mesh**. The price is a **CFL-limited**
  $\Delta t$ (the penalty viscosity is stiff) — but each step is so cheap that **many small
  steps beat a few global solves**, the more so the larger the problem.
- We keep the **same** $D$, so explicit and implicit agree as $\Delta t\to0$: the density still
  stays in $[0,1]$ — the *spatial* scheme kept it bounded, not the implicit damping.

**Projection 1.** The initial crowd is set by an **$L^2$-projection** (`Set`) of the analytic
density onto the order-2 `L2` space.

In [ ]:
audience = 0.55 * 0.5 * (1 + (Hh - 1.0 - y) / sqrt((Hh - 1.0 - y)**2 + 0.3))   # packed behind the tables
speakers = 0.4 * exp(-((x - 4.5)**2 + (y - 5.4)**2) / 0.25)                    # a small group up front
gfu = GridFunction(fes)
gfu.Set(audience + speakers)                          # projection 1: L2-project the initial crowd
mass0 = Integrate(gfu, mesh)
Draw(gfu, mesh, "u")

In [ ]:
Minv = fes.Mass(1).Inverse()                          # the trick: block-diagonal DG mass → element-local
dt = 0.0003                                           # explicit → CFL-limited, but every step is cheap
nsteps = 40000                                        # many cheap steps; run further so more reach coffee

anim_fine = GridFunction(fes,multidim=0)
res = gfu.vec.CreateVector()
snap = max(1, nsteps // 14)

import time
pajetrace = 0            # set e.g. 10**8 to record a Paje timeline of the loop (heavy for many steps)
before = {t["name"]: t["time"] for t in Timers()}     # snapshot the profilers to scope the loop
t0 = time.perf_counter()
with TaskManager(pajetrace=pajetrace):
    for step in range(nsteps):
        C.Apply(gfu.vec, res)                         # explicit convection  C(ρ)
        res.data += D.mat * gfu.vec                   # + explicit viscosity  D·ρ
        gfu.vec.data -= dt * (Minv * res)             # ρ ← ρ − Δt·M⁻¹(C(ρ) + D·ρ)
        if (step + 1) % snap == 0:
            anim_fine.AddMultiDimComponent(gfu.vec)
        progress(step, nsteps)
wall = time.perf_counter() - t0
print(f"{nsteps} steps in {wall:5.1f} s   ({1e3*wall/nsteps:.2f} ms/step)")
delta = sorted(((t["time"] - before.get(t["name"], 0.0), t["name"]) for t in Timers()), reverse=True)
print("hottest routines in the loop:")
for dts, name in delta[:5]:
    if dts > 0:
        print(f"  {dts:7.2f} s  {name[:48]}")

In [ ]:
#Draw(anim_fine,mesh,"sol",interpolate_multidim=True, animate=True, min=0, max=1, autoscale=False)

### Projection 2 — a coarse mesh for the web animation

The fine `multidim` animation is **tens of MB** as a webgui scene — far over what the site can
host. So we **project** each frame onto a **coarse mesh** (interpolate the fine density there):
the same motion, a fraction of the size.

This is also the moment to separate **two knobs** that are easy to confuse:

- the **`order`** of the field (here `L2` order 2 — a quadratic *per element*), and
- the **mesh resolution** `h` — how many elements there are.

We keep the **order** but coarsen the **mesh**; the live notebook can of course still animate
the fine field directly.

In [ ]:
coarse = Mesh(OCCGeometry(room, dim=2).GenerateMesh(maxh=0.45))   # a coarse render mesh
cfes = L2(coarse, order=order)
cframe, ftmp = GridFunction(cfes), GridFunction(fes)
anim = GridFunction(cfes, multidim=0)
for i in range(len(anim_fine.vecs)):
    ftmp.vec.data = anim_fine.vecs[i]
    cframe.Set(ftmp)                                  # projection 2: interpolate fine → coarse
    anim.AddMultiDimComponent(cframe.vec)
Draw(anim, coarse, "crowd density ρ (coarse render)",
     interpolate_multidim=True, animate=True, min=0, max=1, autoscale=False)

## 5. From the crowd to the individual — the same field, one dot at a time

The conservation law tracks a **density** — but every cell of it is made of **people**. The bridge is
direct: drop a few **individual agents** into the *same* field the PDE uses,
$\dot{\mathbf x}_i = v\bigl(\rho(\mathbf x_i,t)\bigr)\,\hat{\mathbf e}(\mathbf x_i)$ — each walks
**along ê** at the **local speed** $v(\rho)=v_{\max}(1-\rho)$. Where the crowd piles up, $\rho\to1$ and
the speed drops to zero: **the individuals slow to a crawl exactly in the jam the continuum predicts.**

*(Pre-rendered offline by `scripts/make_crowd_agents.py`, which runs the very same DG scheme: the
continuum density ρ in blue, agents as dots coloured by speed — green = walking freely, red = jammed.)*

![A few agents advected through the crowd-density field, flowing down the aisles and jamming where the density is high](https://raw.githubusercontent.com/schruste/ngsum2026-colab/colab/data/crowd_agents.gif)

## 6. Going faster — parallelism

- **Threads (shared memory).** The loop already runs in a `TaskManager` → flux applications and
  solves use all cores, with no code change.
- **MPI (distributed memory).** Split the mesh across processes that exchange only their shared
  boundaries; the *same* code runs under `mpirun`:

```python
# mpirun -np 4 python3 crowd_mpi.py
from mpi4py import MPI
from ngsolve import *
comm = MPI.COMM_WORLD
mesh = Mesh(unit_square.GenerateMesh(maxh=0.05, comm=comm))   # distributed mesh
# ... assemble & step exactly as above — NGSolve handles the communication ...
```

In [ ]:
# Navigation between units — shown only in a live notebook (Colab / JupyterLite /
# local Jupyter), never in the rendered website (which has its own prev/next nav).
import os, sys
if not os.environ.get("WEBGUI_SCENE_DIR"):          # not the static site build
    _prev = ("11-turing-patterns", "11 · A different pattern: encounter with a relative 🧬")
    _next = ("13-thermal-plume-hdg", "13 · A puffing thermal plume — HDiv-HDG & HDG 🔥🌀")
    def _u(_nb):
        if "google.colab" in sys.modules:
            return "https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/" + _nb + ".ipynb"
        return _nb + ".ipynb"                       # JupyterLite & local: relative .ipynb link
    _parts  = ["⬅️ **Previous:** [%s](%s)" % (_prev[1], _u(_prev[0]))] if _prev else []
    _parts += ["➡️ **Next:** [%s](%s)" % (_next[1], _u(_next[0]))] if _next else []
    from IPython.display import display, Markdown
    display(Markdown(" · ".join(_parts)))